In [1]:
import pandas as pd

In [2]:
FOLD = 2

In [3]:
flat_windowd_df = pd.read_csv(f"/scratch1/smaruj/genomic_flat_regions/flat_regions_tsv/fold{FOLD}_selected_genomic_windows_centered.tsv", sep="\t")

In [4]:
len(flat_windowd_df)

58

In [5]:
chromhmm_df = pd.read_csv("/home1/smaruj/mESC_mm10_3states_H3K27ac_9ac_9me3_chromHMM.bed", sep="\t", header=None)

In [6]:
chromhmm_df.columns = [
    "chrom",        # Column 1: Chromosome
    "start",        # Column 2: Start coordinate (0-based)
    "end",          # Column 3: End coordinate (exclusive)
    "state",        # Column 4: ChromHMM state (1, 2, or 3)
    "score",        # Column 5: Score (usually 0)
    "strand",       # Column 6: Strand (usually '.')
    "thickStart",   # Column 7: For browser display
    "thickEnd",     # Column 8: For browser display
    "rgb"           # Column 9: Color code for genome browser
]

In [7]:
state_map = {
    1: "active",
    2: "neutral",
    3: "repressive"
}
chromhmm_df["state_label"] = chromhmm_df["state"].map(state_map)

In [8]:
import bioframe as bf

In [9]:
flat_windowd_df = flat_windowd_df.rename(columns={"start": "og_start", "end": "og_end"})

In [10]:
bin_size = 2048

In [11]:
central_bin = 320

In [12]:
# calculating genomic coordinates of the central bin
flat_windowd_df["start"] = flat_windowd_df["centered_start"] + central_bin * bin_size
flat_windowd_df["end"] = flat_windowd_df["centered_start"] + (central_bin + 1) * bin_size

In [13]:
overlap_df = bf.overlap(
    flat_windowd_df,
    chromhmm_df,
    return_index=True,
    suffixes=("_query", "_chromhmm")
)

In [14]:
result = overlap_df[["chrom_query", 
                     "fold_query",
                     "PearsonR_query", 
                     "centered_start_query", 
                     "centered_end_query",
                     "centered_flat_start_query", 
                     "centered_flat_end_query", 
                     "state_chromhmm", 
                     "state_label_chromhmm"
]]

In [15]:
# keep index_query
result = overlap_df[[
    "chrom_query","fold_query","PearsonR_query",
    "centered_start_query","centered_end_query",
    "centered_flat_start_query","centered_flat_end_query",
    "state_label_chromhmm"
]]

In [17]:
grouped = result.groupby(["chrom_query", 
                        "fold_query",
                        "PearsonR_query", 
                        "centered_start_query", 
                        "centered_end_query",
                        "centered_flat_start_query", 
                        "centered_flat_end_query", 
                        "state_label_chromhmm"]).size().reset_index(name="count")

In [19]:
pivoted = grouped.pivot_table(
    index=["chrom_query", 
                     "fold_query",
                     "PearsonR_query", 
                     "centered_start_query", 
                     "centered_end_query",
                     "centered_flat_start_query", 
                     "centered_flat_end_query"],
    columns="state_label_chromhmm",
    values="count",
    fill_value=0
).reset_index()

In [21]:
pivoted.columns.name = None
pivoted = pivoted.rename(columns={
    "active": "active_count",
    "neutral": "neutral_count",
    "repressive": "repressive_count"
})

In [22]:
pivoted

,chrom_query,fold_query,PearsonR_query,centered_start_query,centered_end_query,centered_flat_start_query,centered_flat_end_query,active_count,neutral_count,repressive_count
0,chr11,fold2,0.732358,45981696,47292416,194,318,0.0,2.0,1.0
1,chr11,fold2,0.767461,56039424,57350144,173,339,0.0,1.0,0.0
2,chr11,fold2,0.791830,39716864,41027584,192,320,0.0,1.0,0.0
3,chr11,fold2,0.833249,42188800,43499520,171,341,0.0,1.0,0.0
4,chr11,fold2,0.836212,44619776,45930496,170,342,0.0,1.0,0.0
5,chr11,fold2,0.910790,55146496,56457216,190,322,0.0,1.0,0.0
6,chr11,fold2,0.915895,43421696,44732416,160,352,0.0,1.0,0.0
7,chr14,fold2,0.731114,10399744,11710464,168,344,0.0,1.0,0.0
8,chr14,fold2,0.738000,11220992,12531712,187,325,0.0,1.0,0.0
9,chr14,fold2,0.763067,17168384,18479104,204,308,0.0,1.0,0.0


In [ ]:
pivoted["total"] = pivoted[["active_count", "neutral_count", "repressive_count"]].sum(axis=1)

In [ ]:
for label in ["active", "neutral", "repressive"]:
    pivoted[f"{label}_fraction"] = pivoted[f"{label}_count"] / pivoted["total"]

In [ ]:
columns_to_remove = ["active_count", "neutral_count", "repressive_count", "total"]
pivoted.drop(columns=columns_to_remove, inplace=True)

In [ ]:
pivoted.rename(columns={"chrom_query": "chrom", 
                        "fold_query": "fold",
                        "PearsonR_query": "PearsonR",
                        "centered_start_query": "centered_start",
                        "centered_end_query": "centered_end",
                        "centered_flat_start_query": "centered_flat_start",
                        "centered_flat_end_query": "centered_flat_end"}, inplace=True)

In [ ]:
pivoted

In [ ]:
len(pivoted)

In [ ]:
pivoted.to_csv(f"/scratch1/smaruj/genomic_flat_regions/edited_bin_chrom_states_tsv/fold{FOLD}_edited_bin_chrom_states.tsv", sep="\t", index=False)

In [ ]:
import matplotlib.pyplot as plt

# Sort by one of the fractions (optional)
pivoted_sorted = pivoted.sort_values("active_fraction", ascending=False)

# Plot
fig, ax = plt.subplots(figsize=(12, 6))
bottoms = [0] * len(pivoted_sorted)

for label, color in zip(["active", "neutral", "repressive"], [None, None, None]):
    ax.bar(
        x=range(len(pivoted_sorted)),
        height=pivoted_sorted[f"{label}_fraction"],
        bottom=bottoms,
        label=label
    )
    bottoms = [sum(x) for x in zip(bottoms, pivoted_sorted[f"{label}_fraction"])]

ax.set_xlabel("Genomic Region")
ax.set_ylabel("Chromatin State Fraction")
ax.set_title("Chromatin State Composition per Flat Region")
ax.legend(title="Chromatin State")
ax.set_xticks([])  # Optional: hide x labels if not meaningful
plt.tight_layout()
plt.show()